### Build a QA application over a Graph Database

 Projenin Amacı:

Graph veritabanı üzerinde soru-cevap uygulaması geliştirmek. Kullanıcı doğal dilde soru soruyor, AI Cypher sorgusu oluşturuyor, Neo4j'de çalıştırıyor ve sonucu kullanıcıya döndürüyor.

```markdown
1. Ortam Değişkenlerini Yükleme (import ve dotenv)
    import os: İşletim sistemi ile etkileşim için (dosya yolları, ortam değişkenleri)

    from dotenv import load_dotenv: .env dosyasındaki gizli bilgileri yüklemek için

    load_dotenv(): Proje dizinindeki .env dosyasını okur ve ortam değişkenlerine yükler

    os.getenv("NEO4J_URI"): Ortam değişkenlerinden değerleri okur
```

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

NEO4J_URI =os.getenv("NEO4J_URI")
NEO4J_USERNAME =os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD =os.getenv("NEO4J_PASSWORD")



```markdown
2. Neo4j Graph Database Bağlantısı
Parametrelerin Anlamı:

    url: Neo4j database connection string (neo4j+s://...)

    username: Database kullanıcı adı (genellikle "neo4j")

    password: Database şifresi

    refresh_schema=False: Performans optimizasyonu - schema'yı her seferinde yenilemez

Neo4j Aura Nedir?

    Neo4j'in cloud-based graph database servisi

    neo4j+s:// prefix'i Aura bağlantısını gösterir

Neden refresh_schema=False?

    Schema değişmiyorsa her seferinde yenilemeye gerek yok

    Bağlantı hızını artırır

    Performansı iyileştirir
```

In [2]:
from langchain_community.graphs import Neo4jGraph

#NEO4J_URI = "neo4j+s://0df55adf.databases.neo4j.io"
#NEO4J_USERNAME = "neo4j"
#NEO4J_PASSWORD = "rIbt9q_Z95vAY3-tA9srqbcHRKRAiFhgLEpr2IkKLqw"

# refresh_schema=False ile başlat
graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    refresh_schema=False  # Bu önemli!
)

print("Neo4j Aura bağlantısı başarılı!")
print("Graf nesnesi hazır - şema yükleme atlandı")

# Graph nesnesini kullanabilirsiniz
print("Graph objesi:", graph)

C:\Users\murat\AppData\Local\Temp\ipykernel_29860\2675164990.py:8: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-neo4j package and should be used instead. To use it run `pip install -U :class:`~langchain-neo4j` and import as `from :class:`~langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(


Neo4j Aura bağlantısı başarılı!
Graf nesnesi hazır - şema yükleme atlandı
Graph objesi: <langchain_community.graphs.neo4j_graph.Neo4jGraph object at 0x000001A9DB0B0BC0>


```markdown
3. Veri Yükleme Sorgusu (ETL Process)
CSV Yükleme: LOAD CSV WITH HEADERS FROM 'url' as row

    CSV dosyasını internetten yükler

    Header'ları otomatik olarak algılar

Film Node Oluşturma:
    MERGE: Varsa kullanır, yoksa oluşturur

    SET: Property'leri ayarlar

    date(), toFloat(): Veri tipi dönüşümleri

Yönetmen İlişkileri:
    split(row.director,'|'): "|" ile ayrılmış stringleri diziye çevirir

    trim(director): Boşlukları temizler

    [:DIRECTED_BY]: Yönetmenlik ilişkisi

Aktör İlişkileri: Benzer mantıkla [:ACTED_IN] ilişkisi

Tür İlişkileri: [:HAS_GENRE] ilişkisi
```

In [3]:
# Dataset Moview
moview_query = """
LOAD CSV WITH HEADERS FROM 
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row

MERGE (m:movie{id: row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)

FOREACH (director in split(row.director,'|') | 
    MERGE (p:Person {name: trim(director)})
    MERGE (p)-[:DIRECTED_BY]->(m)
)
FOREACH (actor in split(row.actors,'|') | 
    MERGE (p:Person {name: trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m)
)
FOREACH (genre in split(row.genres,'|') |
    MERGE (g:Genre {name: trim(genre)})
    MERGE (m)-[:HAS_GENRE]->(g)
)
"""
moview_query

"\nLOAD CSV WITH HEADERS FROM \n'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row\n\nMERGE (m:movie{id: row.movieId})\nSET m.released = date(row.released),\n    m.title = row.title,\n    m.imdbRating = toFloat(row.imdbRating)\n\nFOREACH (director in split(row.director,'|') | \n    MERGE (p:Person {name: trim(director)})\n    MERGE (p)-[:DIRECTED_BY]->(m)\n)\nFOREACH (actor in split(row.actors,'|') | \n    MERGE (p:Person {name: trim(actor)})\n    MERGE (p)-[:ACTED_IN]->(m)\n)\nFOREACH (genre in split(row.genres,'|') |\n    MERGE (g:Genre {name: trim(genre)})\n    MERGE (m)-[:HAS_GENRE]->(g)\n)\n"

```markdown
4. Schema Güncelleme ve Görüntüleme
İşlem Sırası:

    graph.query(moview_query): Cypher sorgusunu çalıştırır → verileri yükler

    graph.refresh_schema(): Database'in güncel schema'sını alır

    print(graph.schema): Schema'yı terminalde gösterir
```

In [4]:
graph.query(moview_query)
graph.refresh_schema()
print(graph.schema)

Node properties:
CEO {name: STRING, POB: STRING, YOB: INTEGER}
Employee {name: STRING, POB: STRING, YOB: INTEGER}
Company {name: STRING}
Country {name: STRING}
Person {name: STRING, born: INTEGER}
Movie {title: STRING, released: INTEGER}
movie {id: STRING, title: STRING, released: DATE, imdbRating: FLOAT}
Genre {name: STRING}
Relationship properties:

The relationships:
(:Person)-[:ACTED_IN]->(:movie)
(:Person)-[:DIRECTED_BY]->(:movie)
(:movie)-[:HAS_GENRE]->(:Genre)


```markdown
5. Language Model (LLM) Kurulumu
Groq Nedir?

    Yüksek performanslı AI inference platformu

    Çok hızlı response süreleri

Gemma2-9b-It Modeli:

    Google'ın geliştirdiği 9 milyar parametreli model

    "It" → Instruction-tuned (talimatlara uygun cevaplar için optimize edilmiş)
```

In [5]:
from langchain_groq import ChatGroq
groq_api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="Gemma2-9b-It"
)
llm


ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001A9EF5518E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001A9EF5A9100>, model_name='Gemma2-9b-It', model_kwargs={}, groq_api_key=SecretStr('**********'))

```markdown
6. GraphCypherQAChain Oluşturma
Chain'in Yapısı:

    Cypher Generation Chain: Sorudan Cypher sorgusu üretir

    Query Execution: Sorguyu Neo4j'de çalıştırır

    Response Generation: Sonucu insan diline çevirir

Parametreler:

    verbose=True: Debug bilgilerini gösterir

    allow_dangerous_requests=True: Güvenlik kısıtlamalarını kaldırır (dikkatli kullanın!)
```

In [7]:
from langchain.chains import GraphCypherQAChain
chain = GraphCypherQAChain.from_llm(llm,graph=graph, verbose=True,allow_dangerous_requests=True)
chain

GraphCypherQAChain(verbose=True, graph=<langchain_community.graphs.neo4j_graph.Neo4jGraph object at 0x000001A9DB0B0BC0>, cypher_generation_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['question', 'schema'], input_types={}, partial_variables={}, template='Task:Generate Cypher statement to query a graph database.\nInstructions:\nUse only the provided relationship types and properties in the schema.\nDo not use any other relationship types or properties that are not provided.\nSchema:\n{schema}\nNote: Do not include any explanations or apologies in your responses.\nDo not respond to any questions that might ask anything else than for you to construct a Cypher statement.\nDo not include any text except the generated Cypher statement.\n\nThe question is:\n{question}'), llm=ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001A9EF5518E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001A9EF5A9100>, model_name=

```markdown
7. Sorgu Çalıştırma
Arkaplanda Olanlar:

    Soru: "Who is the director of movie Casino?"

    LLM Cypher Üretir: MATCH (m:movie {title: "Casino"})<-[:DIRECTED_BY]-(p:Person) RETURN p.name

    Neo4j'de Çalıştırır: Sorgu çalışır, sonuçlar alınır

    Formatlama: Sonuçlar kullanıcıya anlaşılır şekilde sunulur
```

In [11]:
response = chain.invoke({
    "query": "Who is the director of movie Casino?"
})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (m:movie {title: "Casino"})<-[:DIRECTED_BY]-(p:Person) RETURN p.name
Full Context:
[{'p.name': 'Martin Scorsese'}]

> Finished chain.


```markdown
Kullanıcı Sorusu 
    → LLM (Cypher oluşturur) 
    → Neo4j (Sorgu çalıştırır) 
    → LLM (Sonucu formatlar) 
    → Kullanıcı Cevabı

Bu yapı sayesinde SQL bilmeyen kullanıcılar doğal dille graph veritabanında sorgulama yapabiliyor!
```